# DeFi-LSA D2 Real Data Collection on Kaggle

This notebook clones a GitHub repository, loads credentials from Kaggle Secrets, installs the package, writes runtime-only source configuration, runs tests, performs connectivity checks, and launches D2 collection.

## Required Kaggle Secrets
- `GITHUB_REPO_URL`: HTTPS URL of the public repository, or a URL that does not embed credentials.
- `GITHUB_TOKEN`: optional, only for a private repository.
- `ETH_RPC_URL`: Ethereum mainnet JSON-RPC endpoint.
- `ETHERSCAN_API_KEY`: Etherscan API key.
- `COINGECKO_API_KEY`: required for CoinGecko price collection. CoinGecko currently rejects unauthenticated historical-range requests with HTTP 401.

Enable Internet in the Kaggle notebook settings before running.

In [4]:
from pathlib import Path
import os, json, subprocess, sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def get_secret(name, required=False):
    try:
        value = secrets.get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing required Kaggle Secret: {name}")
    return value

repo_url = f'https://{get_secret("GITHUB_TOKEN")}@github.com/XQuyTran/DeFi-LSA.git'
repo_branch = 'feature/epic-revised'
os.environ["ETH_RPC_URL"] = get_secret("ETH_RPC_URL", required=True)
os.environ["ETHERSCAN_API_KEY"] = get_secret("ETHERSCAN_API_KEY", required=True)
cg_key = get_secret("COINGECKO_API_KEY")
if cg_key:
    os.environ["COINGECKO_API_KEY"] = cg_key

WORK = Path("/kaggle/working")
REPO = WORK / "defi-lsa"
OUTPUT = WORK / "defi-lsa-output"
restored_candidates = [
    WORK / "defi-lsa-output",
    WORK / "input" / "defi-lsa-output",
]
for candidate in restored_candidates:
    if any((candidate / marker).exists() for marker in ("data", "checkpoints", "reports")):
        OUTPUT = candidate
        break
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Runtime root selected: {OUTPUT}")
print("Secrets loaded; values are not displayed.")

Runtime root selected: /kaggle/working/defi-lsa-output
Secrets loaded; values are not displayed.


In [5]:
import urllib.request
urllib.request.urlopen("https://github.com", timeout=15).read(64)
print("Internet connectivity: OK")


Internet connectivity: OK


In [6]:
import subprocess, os, base64
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", 
                    "--depth", "1", 
                    '--branch', repo_branch, repo_url, str(REPO)], check=True, env=os.environ)
print(subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


Cloning into '/kaggle/working/defi-lsa'...


23c7daed79171644a9ee0a815e28db8903c23650


In [7]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[dev,parquet,aws]"], check=True)
os.environ["PYTHONPATH"] = str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
import defi_lsa
print(f"Package installation: OK ({defi_lsa.__file__})")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
boto3 1.44.4 requires botocore<1.45.0,>=1.44.4, but you have botocore 1.43.117 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.


Package installation: OK (/kaggle/working/defi-lsa/src/defi_lsa/__init__.py)


In [8]:
runtime_config = {
  "coingecko": {
    "enabled": False,
    "allow_keyless": False,
    "base_url": "https://api.coingecko.com/api/v3"
  },
  "etherscan_api_url": "https://api.etherscan.io/v2/api",
  "coingecko_api_url": "https://api.coingecko.com/api/v3",
  "gdelt_enabled": True,
  "request_timeout_seconds": 30,
  "maximum_attempts": 5,
  "requests_per_second": 2.0,
  "initial_log_chunk_size": 2000,
  "minimum_log_chunk_size": 10,
  "maximum_log_chunk_size": 20000
}
config_path = OUTPUT / "runtime_sources.json"
config_path.write_text(json.dumps(runtime_config, indent=2), encoding="utf-8")
print(f"Secret-free runtime configuration written to {config_path}")

Secret-free runtime configuration written to /kaggle/working/defi-lsa-output/runtime_sources.json


In [9]:
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONPATH"] = str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
from defi_lsa.config import load_source_config
from defi_lsa.runtime import preflight_internet, preflight_output_root

runtime_source_config = load_source_config(config_path)
if runtime_source_config.source_enabled("coingecko"):
    raise RuntimeError("CoinGecko must remain disabled for this smoke run")
preflight = preflight_output_root(OUTPUT, minimum_free_bytes=1_000_000_000)
if not preflight["passed"]:
    raise RuntimeError(f"Runtime preflight failed: {preflight}")
internet = preflight_internet("https://github.com")
if not internet["reachable"]:
    raise RuntimeError(f"Internet preflight failed: {internet}")
print({"coingecko_enabled": False, "output_preflight": preflight, "internet_preflight": internet})

{'coingecko_enabled': False, 'output_preflight': {'output_root': '/kaggle/working/defi-lsa-output', 'restored_output': False, 'free_bytes': 20939137024, 'minimum_free_bytes': 1000000000, 'disk_space_ok': True, 'writable': True, 'passed': True}, 'internet_preflight': {'reachable': True, 'status': 200}}


In [10]:
# This command must perform real probes after Epic E2 is implemented.
subprocess.run([
    sys.executable, "-m", "defi_lsa", "connectivity",
    "--event", "euler",
    "--config", str(config_path),
    "--source", "rpc"
], cwd=REPO, check=True)


{"details": {"ethereum_rpc": {}}, "event": "euler", "run_id": "ff795729a59a4980819116221872b8c1", "statuses": {"ethereum_rpc": "reachable"}}


CompletedProcess(args=['/usr/bin/python3', '-m', 'defi_lsa', 'connectivity', '--event', 'euler', '--config', '/kaggle/working/defi-lsa-output/runtime_sources.json', '--source', 'rpc'], returncode=0)

## Smoke collection
Run a small real-data collection before a full event window. The CLI must be wired by Epic E3. If the current repository only writes a summary, stop and implement E3 rather than interpreting the command as a successful collection.


In [11]:
SMOKE_RANGES = {
    "euler": {
        "one_block": (16818024, 16818024),
        "100_blocks": (16817925, 16818024),
    },
    "kyber": {
        "one_block": (18630392, 18630392),
        "100_blocks": (18630293, 18630392),
    },
}
SMOKE_AUDITS = {}
for event, ranges in SMOKE_RANGES.items():
    for label, (start_block, end_block) in ranges.items():
        smoke_output = OUTPUT / "smoke" / f"{event}_{label}"
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "collect",
            "--event", event,
            "--stage", "all",
            "--source", "auto",
            "--start-block", str(start_block),
            "--end-block", str(end_block),
            "--config", str(config_path),
            "--output", str(smoke_output),
            "--resume"
        ], cwd=REPO, check=True)
        audit_event = "kyberswap" if event == "kyber" else event
        audit_result = subprocess.run([
            sys.executable, "-m", "defi_lsa", "audit",
            "--event", audit_event,
            "--stage", "d2",
            "--output", str(smoke_output),
            "--data-dir", str(smoke_output),
            "--strict"
        ], cwd=REPO, capture_output=True, text=True)
        audit_candidates = sorted(smoke_output.rglob("d2_collection_audit.json"))
        if not audit_candidates:
            raise RuntimeError(
                f"Audit report was not created for {event}/{label}; "
                f"returncode={audit_result.returncode}; "
                f"stdout={audit_result.stdout}; stderr={audit_result.stderr}"
            )
        audit_path = audit_candidates[0]
        audit = json.loads(audit_path.read_text(encoding="utf-8"))
        status = audit.get("status")
        if status == "NO_GO":
            raise RuntimeError(f"Smoke audit failed for {event}/{label}: {audit}")
        if status == "CONDITIONAL_GO":
            if runtime_source_config.source_enabled("coingecko"):
                raise RuntimeError(f"Unexpected conditional smoke audit for {event}/{label}: {audit}")
            print(f"Approved CONDITIONAL_GO for {event}/{label}; CoinGecko is disabled by configuration.")
        elif status != "GO":
            raise RuntimeError(f"Unexpected smoke audit status for {event}/{label}: {status}")
        if audit_result.returncode not in (0, 2):
            raise RuntimeError(f"Smoke audit subprocess failed for {event}/{label}: {audit_result.stdout}")
        SMOKE_AUDITS[event, label] = audit_path

{"records_skipped": 0, "records_written": 38, "run_id": "dec63a7ed72f464ea71e48bd02458b55", "status": "completed"}
Approved CONDITIONAL_GO for euler/one_block; CoinGecko is disabled by configuration.
{"records_skipped": 0, "records_written": 98, "run_id": "98f5964f6a7d4dc3bda5aad411f9efdc", "status": "completed"}
Approved CONDITIONAL_GO for euler/100_blocks; CoinGecko is disabled by configuration.
{"records_skipped": 0, "records_written": 438, "run_id": "7f16df8abe0a4afb8718f37b143fd635", "status": "completed"}
Approved CONDITIONAL_GO for kyber/one_block; CoinGecko is disabled by configuration.
{"records_skipped": 0, "records_written": 438, "run_id": "b75e4f66e6374d04932d0c5202909287", "status": "completed"}
Approved CONDITIONAL_GO for kyber/100_blocks; CoinGecko is disabled by configuration.


In [12]:
for (event, label), audit_path in sorted(SMOKE_AUDITS.items()):
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    print({
        "event": event,
        "label": label,
        "status": audit.get("status"),
        "audit_path": str(audit_path),
        "effective_range": audit.get("effective_range"),
    })

{'event': 'euler', 'label': '100_blocks', 'status': 'CONDITIONAL_GO', 'audit_path': '/kaggle/working/defi-lsa-output/smoke/euler_100_blocks/euler_2023-03-13/d2_collection_audit.json', 'effective_range': {'end_block': 16818024, 'overridden': True, 'registry_end_block': 16845224, 'registry_start_block': 16773224, 'start_block': 16817925}}
{'event': 'euler', 'label': 'one_block', 'status': 'CONDITIONAL_GO', 'audit_path': '/kaggle/working/defi-lsa-output/smoke/euler_one_block/euler_2023-03-13/d2_collection_audit.json', 'effective_range': {'end_block': 16818024, 'overridden': True, 'registry_end_block': 16845224, 'registry_start_block': 16773224, 'start_block': 16818024}}
{'event': 'kyber', 'label': '100_blocks', 'status': 'CONDITIONAL_GO', 'audit_path': '/kaggle/working/defi-lsa-output/smoke/kyber_100_blocks/kyberswap_elastic_2023-11-22/d2_collection_audit.json', 'effective_range': {'end_block': 18630392, 'overridden': True, 'registry_end_block': 18630392, 'registry_start_block': 18630392,

## Full event runs
Uncomment only after the smoke audit reports `GO` or an approved `CONDITIONAL_GO`, and after Euler/Curve/Kyber registries contain complete windows, incident legs, token mappings, and public-source queries.


In [13]:
EVENTS = ["euler", "kyber"]
RUN_FULL = True  # Set to True to run full collection and audit after smoke tests
if RUN_FULL:
    for event in EVENTS:
        smoke_audits = [
            path for (audit_event, _label), path in SMOKE_AUDITS.items()
            if audit_event == event
        ]
        if not smoke_audits:
            raise RuntimeError(f"Missing smoke audits for {event}")
        approved_statuses = {"GO", "CONDITIONAL_GO"}
        for audit_path in smoke_audits:
            audit = json.loads(audit_path.read_text(encoding="utf-8"))
            if audit.get("status") not in approved_statuses:
                raise RuntimeError(f"Full run blocked for {event}: {audit.get('status')}")
            if audit.get("status") == "CONDITIONAL_GO" and runtime_source_config.source_enabled("coingecko"):
                raise RuntimeError(f"Unexpected CoinGecko dependency for full {event} run")
        event_output = OUTPUT / "full" / event
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "collect",
            "--event", event,
            "--stage", "all",
            "--source", "auto",
            "--config", str(config_path),
            "--output", str(event_output),
            "--resume"
        ], cwd=REPO, check=True)
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "audit",
            "--event", event,
            "--stage", "d2",
            "--output", str(event_output),
            "--data-dir", str(event_output),
            "--strict"
        ], cwd=REPO, check=True)

{"error": "HTTP Error 400: Bad Request", "run_id": "7cfe96b527b1453882ccd1ed93d556ce", "status": "failed"}


CalledProcessError: Command '['/usr/bin/python3', '-m', 'defi_lsa', 'collect', '--event', 'euler', '--stage', 'all', '--source', 'auto', '--config', '/kaggle/working/defi-lsa-output/runtime_sources.json', '--output', '/kaggle/working/defi-lsa-output/full/euler', '--resume']' returned non-zero exit status 1.

## D3 decoding and classification

Set `RUN_D3 = True` only after the Euler and Kyber D2 audits have completed. D3 runs against isolated event roots because its canonical Parquet filenames are shared within an output root. Curve remains outside this run.

In [ ]:
RUN_D3 = True
D3_EVENTS = {
    "euler": "euler_2023-03-13",
    "kyber": "kyberswap_elastic_2023-11-22",
}
D3_OUTPUTS = [
    "decoded_events.parquet",
    "decoded_calls.parquet",
    "classified_transactions.parquet",
    "actor_roles.parquet",
    "incident_legs_resolved.parquet",
    "balance_inputs.parquet",
]
D3_REPORTS = ["unknown_event_report.json", "d3_gate_report.json"]

if RUN_D3:
    for event_name, event_id in D3_EVENTS.items():
        event_output = OUTPUT / "full" / event_name
        audit_candidates = sorted(event_output.rglob("d2_collection_audit.json"))
        if not audit_candidates:
            raise RuntimeError(f"Missing D2 audit for {event_name}: {event_output}")
        audit_path = audit_candidates[0]
        audit = json.loads(audit_path.read_text(encoding="utf-8"))
        if audit.get("status") not in {"GO", "CONDITIONAL_GO"}:
            raise RuntimeError(f"D3 blocked for {event_name}: {audit.get('status')}")
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "d3",
            "--event", event_id,
            "--output", str(event_output),
            "--events", str(REPO / "configs" / "events.json"),
            "--incident-legs", str(REPO / "configs" / "incident_legs.json"),
        ], cwd=REPO, check=True)
        d3_report_path = event_output / "reports" / event_id / "d3_gate_report.json"
        if not d3_report_path.exists():
            raise RuntimeError(f"D3 gate report was not created for {event_name}")
        missing_outputs = [
            name for name in D3_OUTPUTS
            if not (event_output / "data" / "decoded" / name).exists()
        ]
        missing_reports = [
            name for name in D3_REPORTS
            if not (event_output / "reports" / event_id / name).exists()
        ]
        if missing_outputs or missing_reports:
            raise RuntimeError({
                "event": event_name,
                "missing_outputs": missing_outputs,
                "missing_reports": missing_reports,
            })
        print({
            "event": event_name,
            "d2_status": audit.get("status"),
            "d3_report": str(d3_report_path),
        })

In [ ]:
# Sanity-check generated artifacts without exposing secrets.
for path in sorted(OUTPUT.rglob("*")):
    if path.is_file() and path != config_path:
        print(path.relative_to(OUTPUT), path.stat().st_size)
